# homeassistant / weather / forecast

**Intent**
- Predicts `weather_out`, a scaffold name until the signature is defined, and must:
  - Predict the roof sensor's actual temperature from the Bureau of Meteorology forecast.
  - Adjust the forecast by the recent difference between the roof readings and the forecasts made for them.
  - Also use solar radiation and wind, for how the roof differs from the Bureau's station.

**Data**
- `training` and `validation` are the roof readings with the forecasts issued for them.
- `backtest` is whole recorded days replayed.
- `golden` is fixed samples and a missing input.
- The roof sensor has 19,690 readings and the forecasts 22,488 each, checked 2026-10-31.
- There are about 79 days, from late winter to early summer at one site, so split by whole days.

**Status**
- A placeholder, and the sections below are the stages still to be built.
- The signature is not yet defined, and the scaffold names are `weather_in` and `weather_out`.
- The Bureau's station is not on the roof, so part of the adjustment is real site difference.

**Future**
- Overall: the work still to do before the first release:
  - Build the dataset of realised roof readings paired with the forecast issued for them.
  - Carry the recent error as state, an exponentially weighted bias per horizon.
  - Train with scikit-learn, convert with `skl2onnx`, and replay the backtest through `onnxruntime`.
  - State a golden tolerance for the graph, the smallest its own error on `validation` allows, since float32 kernels differ between CPUs.
  - Set the tolerance for narrowing float64 where an engine lacks a kernel, such as `Atan` in `onnxruntime`.
  - Confirm that `weewx` owns the `roof_*` and `bom_*` entities.
  - Write golden samples, including a missing required input.
- Phase 1: now, collect and prepare the roof data:
  - Collect one run of realised roof readings paired with their forecasts, in time order, so every fold sees days in sequence.
  - Declare `backtest` as the last days with `prepare_split`, and keep it outside every fold.
- Phase 2: cross-validate the learned model with `forward` folds:
  - Use `prepare_folds` with the `forward` scheme over whole days, so each fold trains on earlier days and validates on the next block.
  - Leave a gap of at least the bias memory between training and validation, so the carried error cannot leak across a fold.
  - Fit in each fold, call `fold.record` with its error, and pick the hyperparameters by the mean fold error.
  - Then declare the final splits, fit once on all the earlier days, and replay the backtest through `onnxruntime`.
  - Gate the fold error with checks named `rmse_mean` and `rmse_worst`, since 79 days from winter to summer may hide a failing season.
- Phase 3: as each season arrives:
  - Collect each new season as its own run, so a later phase can hold a whole season out.
  - Repeat the folds on the longer series, and release only when the worst fold stays inside the limit.


## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("homeassistant/weather/forecast")
CONSTANTS = {"weather_out": 1}

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

## Explore ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-explore))

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

## Evaluate ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-evaluate))

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
print("no model is fitted yet, so nothing is packaged")

## Register ([6.7.9](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#679-register))

In [ ]:
print("no model is defined yet, so nothing is registered")